# Chapter 3 notebook — ONNX Runtime inference & benchmark

Walks through `docs/06_onnx_runtime.md`:
1. Load the ONNX file exported in Chapter 5.
2. Run inference on the sample image.
3. Benchmark ORT CPU vs PyTorch CPU on the same model.
4. (If available) switch to CUDA / TensorRT provider and compare.

Run from the repo root.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))   # so we can import src.benchmark

import numpy as np
import onnxruntime as ort
import torch
from PIL import Image
from torchvision import models

from src.benchmark import bench_full, format_report

print('onnxruntime', ort.__version__)
print('available providers:', ort.get_available_providers())

ONNX_PATH = Path('../experiments/exported_models/mobilenet_v3_small.onnx')
assert ONNX_PATH.exists(), f'missing {ONNX_PATH}. Run the Chapter 2 notebook first.'

onnxruntime 1.26.0
available providers: ['AzureExecutionProvider', 'CPUExecutionProvider']


## 1. Load the session and inspect inputs/outputs

In [2]:
opts = ort.SessionOptions()
opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
opts.intra_op_num_threads = 0

sess = ort.InferenceSession(ONNX_PATH.as_posix(), sess_options=opts,
                            providers=['CPUExecutionProvider'])

for i in sess.get_inputs():
    print(f'INPUT  {i.name}  shape={i.shape}  dtype={i.type}')
for o in sess.get_outputs():
    print(f'OUTPUT {o.name}  shape={o.shape}  dtype={o.type}')

INPUT  input  shape=[1, 3, 224, 224]  dtype=tensor(float)
OUTPUT logits  shape=[1, 1000]  dtype=tensor(float)


## 2. Run inference on the sample image

In [3]:
weights = models.MobileNet_V3_Small_Weights.IMAGENET1K_V1
preprocess = weights.transforms()
classes = weights.meta['categories']

img = Image.open('../datasets/sample.jpg').convert('RGB')
x = preprocess(img).unsqueeze(0).numpy()

(logits,) = sess.run(None, {sess.get_inputs()[0].name: x})
probs = np.exp(logits - logits.max()) / np.exp(logits - logits.max()).sum(axis=1, keepdims=True)
top_idx = probs[0].argsort()[::-1][:5]
for i in top_idx:
    print(f'  {probs[0, i] * 100:5.2f}%  {classes[i]}')

  21.64%  candle
  17.40%  jack-o'-lantern
   9.80%  ocarina
   7.91%  orange
   2.65%  pinwheel


## 3. Benchmark ONNX Runtime CPU vs PyTorch CPU

In [4]:
# ONNX Runtime CPU
in_name = sess.get_inputs()[0].name
x_np = np.random.randn(1, 3, 224, 224).astype(np.float32)

def step_ort():
    return sess.run(None, {in_name: x_np})

rpt_ort = bench_full(step_ort, name='mnv3s-ort-cpu-fp32', device='cpu', warmup=10, repeat=50,
                     extras={'runtime': 'onnxruntime', 'provider': 'CPUExecutionProvider'})
print(format_report(rpt_ort))

=== benchmark: mnv3s-ort-cpu-fp32 ===
device=cpu  host=DESKTOP
os=Linux-6.12.90+deb13-amd64-x86_64-with-glibc2.41  python=3.11.15  cpus=16
torch=2.12.0+cu130
extras: runtime=onnxruntime, provider=CPUExecutionProvider

[ model-only latency ]
  warmup=10  repeat=50
  mean=1.50 ms  std=0.34
  P50=1.41  P95=2.12  P99=2.32
  min=1.10  max=2.45
  fps_estimate=668.5

[ memory ]
  cpu_rss_peak=748.0 MB  (+0.0 MB over baseline)


In [5]:
# PyTorch CPU
model = models.mobilenet_v3_small(weights=weights).eval()
x_t = torch.randn(1, 3, 224, 224)

def step_torch():
    with torch.no_grad():
        return model(x_t)

rpt_torch = bench_full(step_torch, name='mnv3s-torch-cpu-fp32', device='cpu', warmup=10, repeat=50,
                       extras={'runtime': 'pytorch', 'provider': 'cpu'})
print(format_report(rpt_torch))

=== benchmark: mnv3s-torch-cpu-fp32 ===
device=cpu  host=DESKTOP
os=Linux-6.12.90+deb13-amd64-x86_64-with-glibc2.41  python=3.11.15  cpus=16
torch=2.12.0+cu130
extras: runtime=pytorch, provider=cpu

[ model-only latency ]
  warmup=10  repeat=50
  mean=4.54 ms  std=0.33
  P50=4.44  P95=5.02  P99=5.70
  min=4.14  max=6.11
  fps_estimate=220.3

[ memory ]
  cpu_rss_peak=774.0 MB  (+0.0 MB over baseline)


In [6]:
# Side-by-side comparison
rows = [
    ('PyTorch CPU', rpt_torch),
    ('ORT CPU',     rpt_ort),
]
print(f'{"runtime":<14} {"P50 ms":>8} {"P95 ms":>8} {"FPS":>8} {"RSS MB":>10}')
for label, r in rows:
    lat = r['latency']; mem = r['memory']
    print(f'{label:<14} {lat["latency_p50_ms"]:>8.2f} {lat["latency_p95_ms"]:>8.2f} '
          f'{lat["fps_estimate"]:>8.1f} {mem["cpu_rss_peak_mb"]:>10.1f}')

speedup = rpt_torch['latency']['latency_p50_ms'] / rpt_ort['latency']['latency_p50_ms']
print(f'\nORT/PyTorch P50 speedup: {speedup:.2f}x')

runtime          P50 ms   P95 ms      FPS     RSS MB
PyTorch CPU        4.44     5.02    220.3      774.0
ORT CPU            1.41     2.12    668.5      748.0

ORT/PyTorch P50 speedup: 3.16x


## 4. Optional: CUDA execution provider

Only runs if `CUDAExecutionProvider` is in `get_available_providers()`. Skip if you do not have `onnxruntime-gpu` installed.

In [7]:
providers = ort.get_available_providers()
if 'CUDAExecutionProvider' in providers:
    sess_cuda = ort.InferenceSession(ONNX_PATH.as_posix(), providers=['CUDAExecutionProvider'])
    in_name_c = sess_cuda.get_inputs()[0].name

    def step_cuda():
        return sess_cuda.run(None, {in_name_c: x_np})

    rpt_cuda = bench_full(step_cuda, name='mnv3s-ort-cuda-fp32', device='cuda',
                          warmup=20, repeat=50,
                          extras={'runtime': 'onnxruntime', 'provider': 'CUDAExecutionProvider'})
    print(format_report(rpt_cuda))
else:
    print('CUDAExecutionProvider not available — skipping. Install onnxruntime-gpu to enable.')

CUDAExecutionProvider not available — skipping. Install onnxruntime-gpu to enable.


## Take-aways

1. ORT on CPU is often **2-5× faster** than PyTorch CPU for small models like MobileNetV3-Small.
2. Predictions match exactly (you verified this in the Chapter 2 notebook).
3. Swapping providers is a one-line change — start on CPU, add CUDA / TensorRT / OpenVINO when the hardware is there.
4. Always rerun `bench_full` after any change so your `experiments/benchmark_results/` history is comparable.